# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [3]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:

EVENT_NAME = '202410_Hurricane_Milton'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'opera'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 11 .tif files in the S3 bucket.


['drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241003.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241008.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_HLS_WTR_20240927_20241007_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241003_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241008_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DS

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [13]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 0
  - Total size: 0.00 GB


(0, 0)

In [14]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [15]:
keys

['drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241003.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241008.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_HLS_WTR_20240927_20241007_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241003_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241008_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DS

In [16]:
# Define filename creator functions for different file types

def create_cog_filename_gen_anom_L9(f, EVENT_NAME):
    """Create COG filename for water mask files with formatted date."""
    from pathlib import Path
    import re
    
    filename_stem = Path(f).stem
    
    # Remove EVENT_NAME from the beginning if it's already there
    if filename_stem.startswith(EVENT_NAME):
        filename_stem = filename_stem[len(EVENT_NAME)+1:]  # +1 to remove the underscore
    
    # Find date pattern (8 digits starting with 20)
    # This will find any 8-digit sequence starting with 20
    date_match = re.search(r'(20\d{6})', filename_stem)
    
    if date_match:
        date_str = date_match.group(1)
        # Format date as YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Replace the date pattern
        if f"{date_str}day" in filename_stem:
            filename_stem = filename_stem.replace(f"{date_str}day", f"{formatted_date}_day")
        else:
            # Just replace the date itself
            filename_stem = filename_stem.replace(date_str, formatted_date)
            # If it doesn't already end with _day, add it
            if not filename_stem.endswith("_day"):
                filename_stem = f"{filename_stem}_day"
    else:
        # If no date found, just add _day at the end if not already there
        if not filename_stem.endswith("_day"):
            filename_stem = f"{filename_stem}_day"
    
    cog_filename = f'{EVENT_NAME}_{filename_stem}.tif'
    return cog_filename


filter_str = 'L9'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_gen_anom_L9(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_2024-10-11_day.tif
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_2024-10-11_day.tif


In [17]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_gen_anom_L9, 
                                target_dir = "HLS/aria", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_2024-10-11_day.tif
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_2024-10-11_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Hurricane_Milton/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/aria

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/2] Processing: drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_20241011.tif
   Output filename: 202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_2024-10-11_day.tif
   [MEMORY] Initial: 293.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 2.00 MB
   [NODATA] Source nodata value: -1.0
   [CHUNKS] Proc

Band 1:  26%|██▋       | 146/551 [00:06<00:18, 21.34chunks/s]


   [MEMORY] High usage: 595.5 MB, forcing cleanup...


Band 1:  28%|██▊       | 153/551 [00:06<00:23, 16.75chunks/s]


   [MEMORY] High usage: 597.5 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=255, max=255, center sample non-zero=0/1000000
            Estimated data coverage: 0.0% (from distributed samples)
   [WARNING] Band 1 appears to have no data after reprojection!
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmppwqt44nh_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpz9691fve.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_2024-10-11_day.tif
   [MEMORY] Final: 1271.0 MB (Change: +78.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_2024-10-11_day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T12:25:30.149340


In [18]:
keys

['drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241003.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241008.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_HLS_WTR_20240927_20241007_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241003_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241008_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DS

In [19]:
# Define filename creator functions for different file types

filter_str = 'MAX_S2'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_gen_anom_L9(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_2024-10-12_day.tif
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-12_day.tif


In [20]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_gen_anom_L9, 
                                target_dir = "Sentinel-2/distAlert", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_2024-10-12_day.tif
  202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-12_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Hurricane_Milton/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-2/distAlert

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/2] Processing: drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_20241012.tif
   Output filename: 202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_2024-10-12_day.tif
   [MEMORY] Initial: 1271.0 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 2.00 MB
   [NODATA] Source nodata value: -1.0
   [CHUNKS]

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-1, max=-1, center sample non-zero=0/1000000
            Estimated data coverage: 0.4% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: int16
   [NODATA] Using nodata value -9999 for int16 data
   [PREDICTOR] Data type: int16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpcetdkj1n_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmppn20x16b.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=255, max=255, center sample non-zero=0/1000000
            Estimated data coverage: 0.4% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpbcl7bwho_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp42pv4ild.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-12_day.tif
   [MEMORY] Final: 1196.5 MB (Change: +0.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_2024-10-12_day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/distAlert/files_converted.csv
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16

In [22]:
def create_cog_filename_chngMap(f, EVENT_NAME):
    """Create COG filename for water mask files, reversing dates for BWTR_ChngMap files."""
    f2 = Path(f).stem
    
    # Check if it's a BWTR_ChngMap file with dates in format YYYYMMDD-YYYYMMDD
    if 'BWTR_ChngMap' in f2 and '-' in f2:
        # Split to get the date part
        parts = f2.split('_')
        for i, part in enumerate(parts):
            if '-' in part and len(part) == 17:  # YYYYMMDD-YYYYMMDD
                date1, date2 = part.split('-')
                
                # Format dates as YYYY-MM-DD
                formatted_date1 = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}"
                formatted_date2 = f"{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
                
                # Reverse the dates and add 'c' prefix
                parts[i] = f'c{formatted_date2}_{formatted_date1}'
                break
        f2 = '_'.join(parts)
    
    cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    return cog_filename

filter_str = 'ChngMap'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_chngMap(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-03_2024-10-11_day.tif
  202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-08_2024-10-11_day.tif


In [23]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_chngMap, 
                                target_dir = "Sentinel-1/opera_dswx", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-03_2024-10-11_day.tif
  202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-08_2024-10-11_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Hurricane_Milton/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/opera_dswx

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/2] Processing: drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241003.tif
   Output filename: 202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-03_2024-10-11_day.tif
   [MEMORY] Initial: 1196.5 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 4.00 MB
   [NODATA] Source nodata value: -3.402823466385288

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-1.0, max=1.0, center sample non-zero=1000000/1000000
            Estimated data coverage: 20.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpqnept0u5_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpj9owlj8f.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-3.4028234663852886e+38, max=-3.4028234663852886e+38, center sample non-zero=0/1000000
            Estimated data coverage: 20.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpr0ieb2tm_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpyx65gorm.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-08_2024-10-11_day.tif
   [MEMORY] Final: 1821.6 MB (Change: -2.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_OPERA_DSWx-S1_BWTR_ChngMap_c2024-10-08_2024-10-11_day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/files_converted.csv
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09

In [24]:
keys


['drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241003.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241008.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_HLS_WTR_20240927_20241007_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241003_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241008_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DS

In [28]:
def create_cog_filename_HLS_WTR(f, EVENT_NAME):
    """Create COG filename for water mask files, moving mosaic before dates and formatting dates."""
    f2 = Path(f).stem
    
    # Check if it has the pattern with _mosaic at the end
    if '_WTR_' in f2 and '_mosaic' in f2:
        # Remove _mosaic from the end
        f2_no_mosaic = f2.replace('_mosaic', '')
        
        # Split by underscore
        parts = f2_no_mosaic.split('_')
        
        # Find WTR index
        wtr_idx = parts.index('WTR')
        
        # Get everything after WTR
        base_parts = parts[:wtr_idx + 1]  # Everything up to and including WTR
        date_parts = parts[wtr_idx + 1:]  # The dates (should be 2 dates in YYYYMMDD format)
        
        # Format the dates
        if len(date_parts) >= 2:
            date1 = date_parts[0]
            date2 = date_parts[1]
            
            # Convert YYYYMMDD to YYYY-MM-DD
            if len(date1) == 8 and date1.isdigit():
                formatted_date1 = f"{date1[:4]}-{date1[4:6]}-{date1[6:8]}"
            else:
                formatted_date1 = date1
                
            if len(date2) == 8 and date2.isdigit():
                formatted_date2 = f"{date2[:4]}-{date2[4:6]}-{date2[6:8]}"
            else:
                formatted_date2 = date2
            
            # Reconstruct with mosaic before dates and 'day' after first date
            f2 = '_'.join(base_parts) + '_mosaic_' + formatted_date1 + 'day_' + formatted_date2
        else:
            # If date pattern doesn't match expected format, just move mosaic
            f2 = '_'.join(base_parts) + '_mosaic_' + '_'.join(date_parts) + 'day'
    else:
        # If pattern doesn't match, just add 'day' at the end
        f2 = f2 + 'day'
    
    cog_filename = f'{EVENT_NAME}_{f2}.tif'
    return cog_filename


filter_str = 'HLS_WTR'    
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_HLS_WTR(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202410_Hurricane_Milton_OPERA_DSWx_HLS_WTR_mosaic_2024-09-27day_2024-10-07.tif


In [29]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_HLS_WTR, 
                                target_dir = "HLS/opera", 
                                EVENT_NAME = EVENT_NAME)




Testing filenames:
  202410_Hurricane_Milton_OPERA_DSWx_HLS_WTR_mosaic_2024-09-27day_2024-10-07.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Hurricane_Milton/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/opera

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/1] Processing: drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_HLS_WTR_20240927_20241007_mosaic.tif
   Output filename: 202410_Hurricane_Milton_OPERA_DSWx_HLS_WTR_mosaic_2024-09-27day_2024-10-07.tif
   [MEMORY] Initial: 1821.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [NODATA] Source nodata value: 255.0
   [CHUNKS] Processing 1794 chunks (46x39)
   [BAND 1/1] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=253, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmphep65cdh_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpxp4g6x88.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/opera/202410_Hurricane_Milton_OPERA_DSWx_HLS_WTR_mosaic_2024-09-27day_2024-10-07.tif
   [MEMORY] Final: 1623.1 MB (Change: -198.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_OPERA_DSWx_HLS_WTR_mosaic_2024-09-27day_2024-10-07.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/opera/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/opera/files_converted.csv
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T12:37:23.306860


In [30]:
keys

['drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_GEN-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_L9S2B_20241011.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dist/ARIA_OPERA-DIST-ALERT_VEG-ANOM-MAX_S2A_20241012.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241003.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx-S1_BWTR_ChngMap_20241011-20241008.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_HLS_WTR_20240927_20241007_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241003_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241008_mosaic.tif',
 'drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DS

In [31]:
from pathlib import Path

def create_cog_filename_S1_WTR(f, EVENT_NAME):
    """Create COG filename for S1 water mask files, moving mosaic and formatting date."""
    f2 = Path(f).stem
    
    # Check if it has the pattern WTR_YYYYMMDD_mosaic
    if '_WTR_' in f2 and '_mosaic' in f2:
        # Remove _mosaic from the end
        f2_no_mosaic = f2.replace('_mosaic', '')
        
        # Split by underscore
        parts = f2_no_mosaic.split('_')
        
        # Find WTR index
        wtr_idx = parts.index('WTR')
        
        # Get the date after WTR
        date_str = parts[wtr_idx + 1]
        
        # Convert YYYYMMDD to YYYY-MM-DD
        if len(date_str) == 8 and date_str.isdigit():
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}_"
        else:
            formatted_date = date_str  # Keep original if not in expected format
        
        # Reconstruct with mosaic before formatted date
        base_parts = parts[:wtr_idx + 1]  # Everything up to and including WTR
        f2 = '_'.join(base_parts) + f'_mosaic_{formatted_date}'
    
    cog_filename = f'{EVENT_NAME}_{f2}day.tif'
    return cog_filename

filter_str = 'S1_WTR'
    
# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_S1_WTR(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-03_day.tif
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-08_day.tif
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-10_day.tif
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-11_day.tif


In [32]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_S1_WTR, 
                                target_dir = "Sentinel-1/opera_dswx", 
                                EVENT_NAME = EVENT_NAME)




Testing filenames:
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-03_day.tif
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-08_day.tif
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-10_day.tif
  202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-11_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Hurricane_Milton/opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/opera_dswx

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/4] Processing: drcs_activations/202410_Hurricane_Milton/opera/dswx/OPERA_DSWx_S1_WTR_20241003_mosaic.tif
   Output filename: 202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-03_day.tif
   [MEMORY] Initial: 1285.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=251, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpxhr3oawb_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp31e_umdp.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=3, center sample non-zero=1000000/1000000
            Estimated data coverage: 60.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpz8xycoap_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp_t1cw7gz.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-11_day.tif
   [MEMORY] Final: 1283.3 MB (Change: +0.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_OPERA_DSWx_S1_WTR_mosaic_2024-10-11_day.tif

✅ Batch processing complete: 4 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/opera_dswx/files_converted.csv
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 4
Successful: 4
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T12:43:50.195819


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")